# Final Project

This final project can be collaborative. The maximum members of a group is 3. You can also work by yourself. Please respect the academic integrity. **Remember: if you get caught on cheating, you get F.**

## A Introduction to the competition

<img src="news-sexisme-EN.jpg" alt="drawing" width="380"/>

Sexism is a growing problem online. It can inflict harm on women who are targeted, make online spaces inaccessible and unwelcoming, and perpetuate social asymmetries and injustices. Automated tools are now widely deployed to find, and assess sexist content at scale but most only give classifications for generic, high-level categories, with no further explanation. Flagging what is sexist content and also explaining why it is sexist improves interpretability, trust and understanding of the decisions that automated tools use, empowering both users and moderators.

This project is based on SemEval 2023 - Task 10 - Explainable Detection of Online Sexism (EDOS). [Here](https://codalab.lisn.upsaclay.fr/competitions/7124#learn_the_details-overview) you can find a detailed introduction to this task.

You only need to complete **TASK A - Binary Sexism Detection: a two-class (or binary) classification where systems have to predict whether a post is sexist or not sexist**. To cut down training time, we only use a subset of the original dataset (5k out of 20k). The dataset can be found in the same folder. 

Different from our previous homework, this competition gives you great flexibility (and very few hints). You can freely determine every component of your workflow, including but not limited to:
-  **Preprocessing the input text**: You may decide how to clean or transform the text. For example, removing emojis or URLs, lowercasing, removing stopwords, applying stemming or lemmatization, correcting spelling, or performing tokenization and sentence segmentation.
-  **Feature extraction and encoding**: You can choose any method to convert text into numerical representations, such as TF-IDF, Bag-of-Words, N-grams, Word2Vec, GloVe, FastText, contextual embeddings (e.g., BERT, RoBERTa, or other transformer-based models), Part-of-Speech (POS) tagging, dependency-based features, sentiment or emotion features, readability metrics, or even embeddings or features generated by large language models (LLMs).
-  **Data augmentation and enrichment**: You may expand or balance your dataset by incorporating other related corpora or using techniques like synonym replacement, random deletion/insertion, or LLM-assisted augmentation (e.g., generating paraphrased or synthetic examples to improve model robustness).
-  **Model selection**: You are free to experiment with different models — from traditional machine learning algorithms (e.g., Logistic Regression, SVM, Random Forest, XGBoost) to deep learning architectures (e.g., CNNs, RNNs, Transformers), or even hybrid/ensemble approaches that combine multiple models or leverage LLM-generated predictions or reasoning.

## Requirements
-  **Input**: the text for each instance.
-  **Output**: the binary label for each instance.
-  **Feature engineering**: use at least 2 different methods to extract features and encode text into numerical values. You may explore both traditional and AI-assisted techniques. Data augmentation is optional.
-  **Model selection**: implement with at least 3 different models and compare their performance.
-  **Evaluation**: create a dataframe with rows indicating feature+model and columns indicating Precision (P), Recall (R) and F1-score (using weighted average). Your results should have at least 6 rows (2 feature engineering methods x 3 models). Report best performance with (1) your feature engineering method, and (2) the model you choose. Here is an example illustrating how the experimental results table should be presented.

| Feature + Model | Sexist (P) | Sexist (R) | Sexist (F1) | Non-Sexist (P) | Non-Sexist (R) | Non-Sexist (F1) | Weighted (P) | Weighted (R) | Weighted (F1) |
|-----------------|:----------:|:----------:|:------------:|:---------------:|:---------------:|:----------------:|:-------------:|:--------------:|:---------------:|
| TF-IDF + Logistic Regression | ... | ... | ... | ... | ... | ... | ... | ... | ... |

- **Format of the report**: add explainations for each step (you can add markdown cells). At the end of the report, write a summary for each sections: 
    - Data Preprocessing
    - Feature Engineering
    - Model Selection and Architecture
    - Training and Validation
    - Evaluation and Results
    - Use of Generative AI (if you use)

## Rules 
Violations will result in 0 points in the grade: 
-   `Rule 1 - No test set leakage`: You must not use any instance from the test set during training, feature engineering, or model selection.
-   `Rule 2 - Responsible AI use`: You may use generative AI, but you must clearly document how it was used. If you have used genAI, include a section titled “Use of Generative AI” describing:
    -   What parts of the project you used AI for
    -   What was implemented manually vs. with AI assistance

## Grading

The performance should be only evaluated on the test set (a total of 1086 instances). Please split original dataset into train set and test set. The test set should NEVER be used in the training process. The evaluation metric is a combination of precision, recall, and f1-score (use `classification_report` in sklearn). 

The total points are 10.0. Each team will compete with other teams in the class on their best performance. Points will be deducted if not following the requirements above. 

If ALL the requirements are met:
- Top 25\% teams: 10.0 points.
- Top 25\% - 50\% teams: 8.5 points.
- Top 50\% - 75\% teams: 7.0 points.
- Top 75\% - 100\% teams: 6.0 points.

If your best performance reaches **0.82** or above (weighted F1-score) and follows all the requirements and rules, you will also get full points (10.0 points). 

## Submission
Similar as homework, submit both a PDF and .ipynb version of the report including: 
- code and experimental results with details explained
- combined results table, report and best performance
- a summary at the end of the report (please follow the format above)

Missing any part of the above requirements will result in point deductions.

The due date is **May 8, Friday by 11:59pm**.

### Imports and Set-Up

In [52]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import warnings
# use 'python -m pip install nltk' im the terminal to install 
import nltk
from nltk.stem import WordNetLemmatizer 
from nltk.tokenize import word_tokenize
nltk.download('wordnet')
nltk.download('punkt_tab')
# user 'pip install sentence-transformers' in the terminal to install
from sentence_transformers import SentenceTransformer
# user 'pip install xgboost' in the terminal to install
from xgboost import XGBClassifier
import re
# Suppress warnings
warnings.filterwarnings("ignore")
from IPython.display import display

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\iyans\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\iyans\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


### Preprocessing: lowercasing, lemmatization, 

In [53]:
data_df = pd.read_csv('edos_labelled_data.csv')
train_df = data_df[data_df['split'].str.contains("train", na=False)].copy()
test_df = data_df[data_df['split'].str.contains("test", na=False)].copy()


def preprocess_text(text):
    text = text.lower()                         # lowercase string
    text = re.sub(r'[^\w\s]', ' ', text)        # remove punctuation
    return text.strip()

# creates a new column with the preprocessed text
train_df['clean_text'] = train_df['text'].apply(preprocess_text)
test_df['clean_text'] = test_df['text'].apply(preprocess_text)
print(train_df[['text', 'clean_text']].head())

# set target values
y_test = test_df['label']
y_train = train_df['label']

                                                text  \
0  In Nigeria, if you rape a woman, the men rape ...   
1                            Then, she's a keeper. 😉   
2  This is like the Metallica video where the poo...   
3                                             woman?   
4                     I bet she wished she had a gun   

                                          clean_text  
0  in nigeria  if you rape a woman  the men rape ...  
1                               then  she s a keeper  
2  this is like the metallica video where the poo...  
3                                              woman  
4                     i bet she wished she had a gun  


### Feature Extraction: Embeddings

In [54]:
# Use SBERT transformer model, specialized for full sentence encoding, to create embeddings for both training and testing. 

sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode with SBERT
x_train_sbert_embeds = sbert_model.encode(train_df['clean_text'].tolist(), show_progress_bar=True).tolist()
x_test_sbert_embeds = sbert_model.encode(test_df['clean_text'].tolist(), show_progress_bar=True).tolist()


Batches: 100%|██████████| 34/34 [00:03<00:00, 11.10it/s]


### Feature Extraction: TF-IDF

In [55]:
# using Tf idf vectorizor to turn the text into a statistical measurement
# it will tokenize and apply weight to words. Tf = term frequency showing
# how often a word appears, and idf = Inverse Document Frequency to show 
# word rarity in the appearance. It is used to filter out words that cause
# noise like stopwords. It will rank them lower rather than remove them.
from sklearn.feature_extraction.text import TfidfVectorizer
word_vec = TfidfVectorizer(ngram_range=(1,2))

x_train_vec = word_vec.fit_transform(train_df['clean_text'])
x_test_vec = word_vec.transform(test_df['clean_text'])

from sklearn.pipeline import Pipeline, FeatureUnion

# vectorize at both a word and character level, then combine, to gain extra insight
vec_word = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
vec_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 3), min_df=2)

combined_features = FeatureUnion([('word', vec_word),
                                  ('char', vec_char)])



### Models: Logistic Regression

In [56]:
from sklearn.linear_model import LogisticRegression

# Logistic Regression Model with SBERT
LR_model_SBERT = LogisticRegression(max_iter=1000)
LR_model_SBERT.fit(x_train_sbert_embeds, y_train)

LR_SBERT_y_pred = LR_model_SBERT.predict(x_test_sbert_embeds)

print("Logistic Regression Model with sbert embeddings")
print(classification_report(y_test, LR_SBERT_y_pred))


# Logistic Regression Model with TFIDF
LR_Model_TFIDF_pipeline = Pipeline([('features', combined_features),
                                    ('clf', LogisticRegression())])

LR_Model_TFIDF_pipeline.fit(train_df['clean_text'], y_train)
LR_TFIDF_y_pred = LR_Model_TFIDF_pipeline.predict(test_df['clean_text'])

print("Logistic Regression Model with TFID vectorization")
print(classification_report(y_test, LR_TFIDF_y_pred))





Logistic Regression Model with sbert embeddings
              precision    recall  f1-score   support

  not sexist       0.82      0.90      0.86       789
      sexist       0.63      0.46      0.53       297

    accuracy                           0.78      1086
   macro avg       0.73      0.68      0.70      1086
weighted avg       0.77      0.78      0.77      1086

Logistic Regression Model with TFID vectorization
              precision    recall  f1-score   support

  not sexist       0.81      0.96      0.88       789
      sexist       0.79      0.40      0.53       297

    accuracy                           0.81      1086
   macro avg       0.80      0.68      0.71      1086
weighted avg       0.80      0.81      0.78      1086



### Models: Support Vector Classifier (SVC)

In [57]:
from sklearn.svm import LinearSVC

## Testing SVC with the SBERT features
SVCModel_sbert = LinearSVC(C=0.35)
SVCModel_sbert.fit(x_train_sbert_embeds, y_train)
svc_sbert_y_pred = SVCModel_sbert.predict(x_test_sbert_embeds)
print("SVC Model with sbert embeddings")
print(classification_report(y_test, svc_sbert_y_pred))

## Testing SVC with the vectorizer features
SVCModel_TfIDF_pipeline = Pipeline([('features', combined_features),
                                    ('clf', LinearSVC(C=0.5))])

SVCModel_TfIDF_pipeline.fit(train_df['clean_text'], y_train)
svc_tfidf_y_pred = SVCModel_TfIDF_pipeline.predict(test_df['clean_text'])
print("SVC Model with TFIDF vectorization")
print(classification_report(y_test, svc_tfidf_y_pred))


SVC Model with sbert embeddings
              precision    recall  f1-score   support

  not sexist       0.82      0.89      0.85       789
      sexist       0.62      0.49      0.55       297

    accuracy                           0.78      1086
   macro avg       0.72      0.69      0.70      1086
weighted avg       0.77      0.78      0.77      1086

SVC Model with TFIDF vectorization
              precision    recall  f1-score   support

  not sexist       0.84      0.93      0.89       789
      sexist       0.75      0.54      0.63       297

    accuracy                           0.83      1086
   macro avg       0.80      0.74      0.76      1086
weighted avg       0.82      0.83      0.82      1086



### Models: XGBoost 

In [58]:
def get_binary_representation(text):
    if text == 'sexist':
        return 1;
    return 0;

params = {
    'objective':'binary:logistic',
    'max_depth':4,
    'learning_rate':0.1,
    'n_estimators':100,
    'alpha':10
}

y_train_binary = y_train.apply(get_binary_representation)
y_test_binary = y_test.apply(get_binary_representation)


model = XGBClassifier(**params)

# SBERT Embeddings
model.fit(x_train_sbert_embeds, y_train_binary)
xg_sbert_y_pred = model.predict(x_test_sbert_embeds)
print("XGBOOST Model with sbert embeddings")
print(classification_report(y_test_binary, xg_sbert_y_pred))

# TFID Vectorization
model.fit(x_train_vec, y_train_binary)
xg_tfidf_y_pred = model.predict(x_test_vec)
print("XGBOOST Model with TFID vectorization")
print(classification_report(y_test_binary, xg_tfidf_y_pred))




XGBOOST Model with sbert embeddings
              precision    recall  f1-score   support

           0       0.79      0.92      0.85       789
           1       0.62      0.36      0.46       297

    accuracy                           0.76      1086
   macro avg       0.70      0.64      0.65      1086
weighted avg       0.74      0.76      0.74      1086

XGBOOST Model with TFID vectorization
              precision    recall  f1-score   support

           0       0.79      0.97      0.87       789
           1       0.79      0.33      0.47       297

    accuracy                           0.79      1086
   macro avg       0.79      0.65      0.67      1086
weighted avg       0.79      0.79      0.76      1086



## Experimental Results

(A table detailed model performance on the test set with at least 6 rows. Report the best performance.)


In [59]:
result_table = pd.DataFrame(columns=['Feature + Model' , 
                                     'Sexist (P)' , 
                                     'Sexist (R)' , 
                                     'Sexist (F1)' , 
                                     'Non-Sexist (P)' , 
                                     'Non-Sexist (R)' , 
                                     'Non-Sexist (F1)' , 
                                     'Weighted (P)' , 
                                     'Weighted (R)' , 
                                     'Weighted (F1)'])

lr_sbert_report = pd.DataFrame(classification_report(y_test, LR_SBERT_y_pred, output_dict=True)).transpose()
lr_sbert_row = pd.DataFrame([['SBERT + Logistic Regression',
                  lr_sbert_report['precision']['sexist'], 
                  lr_sbert_report['recall']['sexist'], 
                  lr_sbert_report['f1-score']['sexist'],
                  lr_sbert_report['precision']['not sexist'], 
                  lr_sbert_report['recall']['not sexist'], 
                  lr_sbert_report['f1-score']['not sexist'],
                  lr_sbert_report['precision']['weighted avg'], 
                  lr_sbert_report['recall']['weighted avg'], 
                  round(lr_sbert_report['f1-score']['weighted avg'],2)]], columns=result_table.columns)
result_table = pd.concat([result_table, lr_sbert_row], ignore_index=True)

lr_tfidf_report = pd.DataFrame(classification_report(y_test, LR_TFIDF_y_pred, output_dict=True)).transpose()
lr_tfidf_row = pd.DataFrame([['TF-IDF + Logistic Regression',
                  lr_tfidf_report['precision']['sexist'], 
                  lr_tfidf_report['recall']['sexist'], 
                  lr_tfidf_report['f1-score']['sexist'],
                  lr_tfidf_report['precision']['not sexist'], 
                  lr_tfidf_report['recall']['not sexist'], 
                  lr_tfidf_report['f1-score']['not sexist'],
                  lr_tfidf_report['precision']['weighted avg'], 
                  lr_tfidf_report['recall']['weighted avg'], 
                  round(lr_tfidf_report['f1-score']['weighted avg'],2)]], columns=result_table.columns)
result_table = pd.concat([result_table, lr_tfidf_row], ignore_index=True)

svc_sbert_report = pd.DataFrame(classification_report(y_test, svc_sbert_y_pred, output_dict=True)).transpose()
svc_sbert_row = pd.DataFrame([['SBERT + SVC',
                  svc_sbert_report['precision']['sexist'], 
                  svc_sbert_report['recall']['sexist'], 
                  svc_sbert_report['f1-score']['sexist'],
                  svc_sbert_report['precision']['not sexist'], 
                  svc_sbert_report['recall']['not sexist'], 
                  svc_sbert_report['f1-score']['not sexist'],
                  svc_sbert_report['precision']['weighted avg'], 
                  svc_sbert_report['recall']['weighted avg'], 
                  round(svc_sbert_report['f1-score']['weighted avg'],2)]], columns=result_table.columns)
result_table = pd.concat([result_table, svc_sbert_row], ignore_index=True)

svc_tfidf_report = pd.DataFrame(classification_report(y_test, svc_tfidf_y_pred, output_dict=True)).transpose()
svc_tfidf_row = pd.DataFrame([['TF-IDF + SVC',
                  svc_tfidf_report['precision']['sexist'], 
                  svc_tfidf_report['recall']['sexist'], 
                  svc_tfidf_report['f1-score']['sexist'],
                  svc_tfidf_report['precision']['not sexist'], 
                  svc_tfidf_report['recall']['not sexist'], 
                  svc_tfidf_report['f1-score']['not sexist'],
                  svc_tfidf_report['precision']['weighted avg'], 
                  svc_tfidf_report['recall']['weighted avg'], 
                  round(svc_tfidf_report['f1-score']['weighted avg'],2)]], columns=result_table.columns)
result_table = pd.concat([result_table, svc_tfidf_row], ignore_index=True)

xg_sbert_report = pd.DataFrame(classification_report(y_test_binary, xg_sbert_y_pred, output_dict=True)).transpose()
xg_sbert_row = pd.DataFrame([['SBERT + XGBoost',
                  xg_sbert_report['precision']['1'], 
                  xg_sbert_report['recall']['1'], 
                  xg_sbert_report['f1-score']['1'],
                  xg_sbert_report['precision']['0'], 
                  xg_sbert_report['recall']['0'], 
                  xg_sbert_report['f1-score']['0'],
                  xg_sbert_report['precision']['weighted avg'], 
                  xg_sbert_report['recall']['weighted avg'], 
                  round(xg_sbert_report['f1-score']['weighted avg'],2)]], columns=result_table.columns)
result_table = pd.concat([result_table, xg_sbert_row], ignore_index=True)

xg_tfidf_report = pd.DataFrame(classification_report(y_test_binary, xg_tfidf_y_pred, output_dict=True)).transpose()
xg_tfidf_row = pd.DataFrame([['TF-IDF + XGBoost',
                  xg_tfidf_report['precision']['1'], 
                  xg_tfidf_report['recall']['1'], 
                  xg_tfidf_report['f1-score']['1'],
                  xg_tfidf_report['precision']['0'], 
                  xg_tfidf_report['recall']['0'], 
                  xg_tfidf_report['f1-score']['0'],
                  xg_tfidf_report['precision']['weighted avg'], 
                  xg_tfidf_report['recall']['weighted avg'], 
                  round(xg_tfidf_report['f1-score']['weighted avg'],2)]], columns=result_table.columns)
result_table = pd.concat([result_table, xg_tfidf_row], ignore_index=True)
display(result_table)


,Feature + Model,Sexist (P),Sexist (R),Sexist (F1),Non-Sexist (P),Non-Sexist (R),Non-Sexist (F1),Weighted (P),Weighted (R),Weighted (F1)
0,SBERT + Logistic Regression,0.634259,0.461279,0.534113,0.816092,0.899873,0.855937,0.766364,0.779926,0.77
1,TF-IDF + Logistic Regression,0.789474,0.40404,0.534521,0.810493,0.959442,0.8787,0.804744,0.807551,0.78
2,SBERT + SVC,0.621277,0.491582,0.548872,0.822562,0.887199,0.853659,0.767514,0.779006,0.77
3,TF-IDF + SVC,0.748837,0.542088,0.628906,0.843858,0.931559,0.885542,0.817871,0.825046,0.82
4,SBERT + XGBoost,0.617143,0.363636,0.457627,0.792536,0.915082,0.849412,0.744569,0.764273,0.74
5,TF-IDF + XGBoost,0.785714,0.333333,0.468085,0.79375,0.965779,0.871355,0.791552,0.792818,0.76


## Project Summary
### 1. Data Preprocessing
We use lowercasing the text and using lemmatization to preprocess the text. We want to avoid capital letters being seen as something different. We also wanted the words to be grouped together when using different endings.

### 2. Feature Engineering
 Our feature engineering came down to SBERT embeddings and TF-IDF. The TF-IDF came from the example to be used with the Logistic Regression. This was done to find a baseline of what worked.
 Later, in order to improve TF-IDF there was the ability to combine the results of word ngrams and char ngrams so the models make a decision using both matrices together.

### 3. Model Selection and Architecture
We have the Logistic Regression, Support Vector Classification (SVC), and [PLACEHOLDER].
All the models will be used with the two feature extraction methods we have choosen.
Logistic Regression was choosen because of the example shown to us. That gave us a baseline to work with in terms of what results it would give. It has also been used for spam and appeared to be solid choice to binary classification of text. Since we already had a TF-IDF feature extractor, it was found that a LinearSVC model would work very well together. The LinearSVC model has shown to output the required weighted average at 0.82 because of the TF-IDF using both word ngrams and char ngrams.

### 4. Training and Validation
Models are given specific training parameters to increase quality of predictions and reduce run-time
Logistic Regression is capped at 1000 iterations
SVM/SVC's regularization parameter is set to c=0.35 to balance training
XGBoost's needs it targets changed to binary values and takes a number of parameters including max_depth

### 5. Evaluation and Results
Our best performing model is the SVC model using the TF-IDF vectorization. It had a Weighted Precision of 0.82, Weighted Recall of 0.83, Weighted F1-Score of 0.82. The worst model was the XGBoost with SBERT encoding. It only reached a Weighted Precision of 0.74, Weighted Recall of 0.76, Weighted F1-Score of 0.74.

### 6. Use of Generative AI (if you use)
Google's AI giving summaries to each of the models and extraction features with generic code outlines. For example I would look up "how to combine tfidf word ngrams with tfidf char ngrams" to put them together. 